## Build and Train the Model

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [2]:
torch.manual_seed(58)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"{device=}")

device='cpu'


This code initializes a tabular dataset of Indonesian dishes along with their flavor profiles (Sour, Sweet, Salty, Spicy) and binary preference labels. It then extracts the 4-dimensional feature array and target vector, standardizes the feature values using StandardScaler, and converts both into PyTorch tensors suitable for neural network input.

In [3]:
###########################################################################
# Dataset
###########################################################################

data = {
    'Name': [
        'Nasi Tim Ayam',
        'Telur Balado',
        'Sayur Bening Bayam',
        'Ayam Goreng Mentega',
        'Tumis Buncis Wortel',
        'Opor Ayam'
    ],
    'Sour': [1, 2, 2, 2, 1, 1],
    'Sweet': [6, 4, 3, 8, 4, 5],
    'Salty': [5, 6, 4, 6, 4, 5],
    'Spicy': [1, 7, 1, 2, 2, 2],
    # Target: 1 for Like, 0 for Dislike
    'Label': [1, 0, 0, 1, 0, 1]
}
df = pd.DataFrame(data)

# Features (Sour, Sweet, Salty, Spicy) -> 4 input dimensions
X = df[['Sour', 'Sweet', 'Salty', 'Spicy']].values
y = df['Label'].values

# Standardize feature values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert to PyTorch tensors
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)  # Shape: (N, 1)

This code defines a lightweight Artificial Neural Network (FoodPreferenceANN) consisting of an input layer, one hidden layer with 8 neurons and ReLU activation, and a single linear output layer. It then instantiates the model with a 4-dimensional input feature space and calculates the total and trainable parameters to print the model's structural footprint.

In [4]:
###########################################################################
# Build neural network model
###########################################################################

class FoodPreferenceANN(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=8):
        super(FoodPreferenceANN, self).__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out = self.layer1(x)
        out = self.relu(out)
        out = self.layer2(out)
        return out

model = FoodPreferenceANN(input_dim=4, hidden_dim=8)
print(model)

# Total parameters
total_params = sum(p.numel() for p in model.parameters())

# Trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params}")
print(f"Trainable Parameters: {trainable_params}")

FoodPreferenceANN(
  (layer1): Linear(in_features=4, out_features=8, bias=True)
  (relu): ReLU()
  (layer2): Linear(in_features=8, out_features=1, bias=True)
)
Total Parameters: 49
Trainable Parameters: 49


This code sets up a binary classification training pipeline using BCEWithLogitsLoss and the Adam optimizer. It then runs a 100-epoch training loop that performs forward passes, computes loss, resets gradients, executes backpropagation, and logs loss progress every 10 epochs.

In [5]:
###########################################################################
# Training
###########################################################################

# Loss Function & Optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training Loop
epochs = 100
verbose_interval = 10  # Print loss every 10 epochs

print("--- Training Started ---")
for epoch in range(1, epochs + 1):
    model.train()
    outputs = model(X_tensor)
    loss = criterion(outputs, y_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Verbose training logs
    if epoch % verbose_interval == 0 or epoch == 1:
        print(f"Epoch [{epoch:3d}/{epochs}] | Loss: {loss.item():.4f}")

print("--- Training Completed ---\n")

--- Training Started ---
Epoch [  1/100] | Loss: 0.7455
Epoch [ 10/100] | Loss: 0.6047
Epoch [ 20/100] | Loss: 0.4555
Epoch [ 30/100] | Loss: 0.3140
Epoch [ 40/100] | Loss: 0.1997
Epoch [ 50/100] | Loss: 0.1207
Epoch [ 60/100] | Loss: 0.0743
Epoch [ 70/100] | Loss: 0.0488
Epoch [ 80/100] | Loss: 0.0350
Epoch [ 90/100] | Loss: 0.0267
Epoch [100/100] | Loss: 0.0209
--- Training Completed ---



This code switches the model to evaluation mode (model.eval()) and disables gradient computation (torch.no_grad()) to run inference on the input dataset. It converts the raw network outputs into probabilities using a sigmoid activation, applies a 0.5 threshold to classify each dish as "Like" or "Dislike", and logs individual predictions alongside the overall dataset accuracy.

In [6]:
###########################################################################
# Inference
###########################################################################

model.eval()
with torch.no_grad():
    raw_logits = model(X_tensor)
    probabilities = torch.sigmoid(raw_logits)
    predictions = (probabilities >= 0.5).float()

print("\n--- Predictions ---")
for name, prob, pred in zip(df['Name'], probabilities, predictions):
    taste = "Like" if pred.item() == 1.0 else "Dislike"
    print(f"Food: {name:22s} | Prob(Like): {prob.item():.4f} | Predicted: {taste}")

# --- Calculate Accuracy ---
correct = (predictions == y_tensor).sum().item()
total = y_tensor.size(0)
accuracy = (correct / total) * 100

print(f"\nAccuracy: {accuracy:.2f}% ({int(correct)}/{total} correct)")


--- Predictions ---
Food: Nasi Tim Ayam          | Prob(Like): 0.9892 | Predicted: Like
Food: Telur Balado           | Prob(Like): 0.0067 | Predicted: Dislike
Food: Sayur Bening Bayam     | Prob(Like): 0.0003 | Predicted: Dislike
Food: Ayam Goreng Mentega    | Prob(Like): 0.9961 | Predicted: Like
Food: Tumis Buncis Wortel    | Prob(Like): 0.0387 | Predicted: Dislike
Food: Opor Ayam              | Prob(Like): 0.9408 | Predicted: Like

Accuracy: 100.00% (6/6 correct)


In [7]:
###########################################################################
# Save weights to a file
###########################################################################

torch.save(model.state_dict(), 'ai_food_weights.pt')
print("Model weights successfully saved to ai_food_weights.pt")

Model weights successfully saved to ai_food_weights.pt
